## ⚠ Save a copy to your Drive first

**This notebook is fetched fresh from GitHub every time you open the link.** Any edits you make here — settings, code, hyperparameters — **will be LOST when you close the tab** unless you save a copy.

**To keep your edits:**

1. **File → Save a copy in Drive** (top menu)
2. Re-open the saved copy via **File → Open notebook → Recent** or your Google Drive next time

The saved copy is yours to edit; the GitHub link always opens fresh.

# Train a Tabular Regressor — RandomForest (simple)

**Maintained by:** IGNODE  
**Last verified:** May 2026 against scikit-learn 1.5  
**Runtime:** under 1 minute on Colab's free CPU tier

Train a tabular regression model using **RandomForest** — interpretable and robust to outliers. Comes with feature-importance ranking. Linear notebook (no branching, no AutoML).

If you want algorithm comparison or AutoML, use `train_tabular_regression.ipynb`.

## Output

`model.onnx` + `feature_columns.json` — drop into IGNODE → ML Factory → Custom Models → + Upload ML Model.

## Quick start

### To try it right now (no setup needed)

1. **Runtime → Run all** at the top of Colab
2. Wait ~30 seconds for dependencies + ~10 seconds for training
3. The last cell automatically downloads `model.onnx` + the sidecar JSON to your laptop — that's a trained RandomForest regressor on the sample IoT data

### To train on YOUR data

You only need to edit **two values**:

| Step | Cell | What to change |
|---|---|---|
| 1 | **Load data** cell (below) | `SAMPLE_DATASET = 'equipment_rul_regression'` → `SAMPLE_DATASET = None` |
| 2 | **Settings** cell | `LABEL_COLUMN = 'RemainingLife'` → `LABEL_COLUMN = 'your_numeric_target_column'` |

The target column **must be numeric**.

### What you get at the end

- `model.onnx` — your trained regression model
- `feature_columns.json` — input contract (`{feature_columns, label_columns}`)
- **Bonus** — RandomForest prints **feature importance ranking** so you can see which features the model relies on most

Drop the artifacts into **IGNODE → ML Factory → Custom Models → + Upload ML Model**.

## 1. Install pinned dependencies

In [ ]:
# Install the libraries this notebook needs.
#
# `--quiet` is omitted on purpose — install errors are visible here
# instead of becoming ModuleNotFoundError later.
#
# onnx-ecosystem packages are NOT pinned (these libraries co-evolve; exact
# pins break with each release). scikit-learn IS pinned for reproducibility.
!pip install scikit-learn==1.5.2 \
             onnx skl2onnx onnxconverter-common

# Verify everything imports cleanly. Failed installs fail fast here
# instead of much later in the export cell.
import sklearn
import skl2onnx  # noqa: F401 — used by the export cell
print(f'scikit-learn: {sklearn.__version__}')

## 2. Load data

Ships set to a small IoT sample (`equipment_rul_regression.csv`) so the notebook runs end-to-end out of the box. To use your own CSV, set `SAMPLE_DATASET = None` below and update `LABEL_COLUMN` in the Settings cell to your numeric target column.

In [ ]:
# ───────── EDIT THIS ─────────
SAMPLE_DATASET = 'equipment_rul_regression'   # set to None to upload your own CSV
# Available samples (regression):
#   'equipment_rul_regression'   — equipment telemetry, label='RemainingLife'
#   'building_energy_regression' — building features, label='HeatingLoad'
# ────────────────────────────

import pandas as pd

if SAMPLE_DATASET:
    url = f'https://raw.githubusercontent.com/IGNODE-CONNECT/ignode-collab/main/examples/{SAMPLE_DATASET}.csv'
    df = pd.read_csv(url)
    print(f'Loaded sample {SAMPLE_DATASET!r}: {df.shape[0]} rows x {df.shape[1]} columns')
else:
    from google.colab import files
    uploaded = files.upload()
    csv_path = next(iter(uploaded.keys()))
    df = pd.read_csv(csv_path)
    print(f'Loaded {csv_path}: {df.shape[0]} rows x {df.shape[1]} columns')

display(df.head())

## 3. Settings

In [ ]:
# ───────── EDIT THESE ─────────
LABEL_COLUMN = 'RemainingLife'   # sample default; change when you bring your own CSV

N_ESTIMATORS = 200
MAX_DEPTH = None          # None = grow until each leaf is pure
MIN_SAMPLES_LEAF = 1

TEST_SIZE = 0.2
RANDOM_SEED = 42
# ──────────────────────────────

if LABEL_COLUMN not in df.columns:
    raise ValueError(f"Label column '{LABEL_COLUMN}' not in CSV. Available: {list(df.columns)}")
if not pd.api.types.is_numeric_dtype(df[LABEL_COLUMN]):
    raise ValueError(
        f"Label column '{LABEL_COLUMN}' must be numeric for regression. "
        f"Got dtype {df[LABEL_COLUMN].dtype}. Use the classifier notebook instead."
    )

## 4. Prep the data

In [ ]:
import re
from sklearn.model_selection import train_test_split

def normalize(name):
    return re.sub(r'[^A-Za-z0-9_-]+', '_', name).strip('_')

rename_map = {c: normalize(c) for c in df.columns if c != normalize(c)}
if rename_map:
    print('Renamed columns:')
    for old, new in rename_map.items():
        print(f'  {old!r}  ->  {new!r}')
    df = df.rename(columns=rename_map)
    if LABEL_COLUMN in rename_map:
        LABEL_COLUMN = rename_map[LABEL_COLUMN]

X = df.drop(columns=[LABEL_COLUMN])
y = df[LABEL_COLUMN].astype(float).values
feature_columns = list(X.columns)

print(f'Features ({len(feature_columns)}): {feature_columns}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED
)
print(f'Train: {X_train.shape[0]} rows. Test: {X_test.shape[0]} rows.')

## 5. Train

In [ ]:
import time
from sklearn.ensemble import RandomForestRegressor

t0 = time.time()
model = RandomForestRegressor(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
model.fit(X_train, y_train)
print(f'Trained in {time.time() - t0:.1f} sec')

## 6. Evaluate

RMSE / MAE / R² + predicted-vs-actual scatter + feature importance ranking.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'Test-set RMSE: {rmse:.3f}')
print(f'Test-set MAE:  {mae:.3f}')
print(f'Test-set R²:   {r2:.3f}')

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, alpha=0.6, s=20)
lim = (min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max()))
ax.plot(lim, lim, 'r--', linewidth=1, label='perfect prediction')
ax.set_xlabel(f'Actual {LABEL_COLUMN}')
ax.set_ylabel(f'Predicted {LABEL_COLUMN}')
ax.set_title(f'Predicted vs Actual (test set, R²={r2:.3f})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Feature importances
importances = model.feature_importances_
order = np.argsort(importances)[::-1]
print('\nFeature importances (most → least useful):')
for idx in order:
    print(f'  {feature_columns[idx]:<30} {importances[idx]:.4f}')

## 7. Export to ONNX

In [ ]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnx

initial_types = [('input', FloatTensorType([None, len(feature_columns)]))]

onnx_model = None
for opset in (18, 15, 12):
    try:
        onnx_model = convert_sklearn(model, initial_types=initial_types, target_opset=opset)
        print(f'  ONNX exported at opset {opset}')
        break
    except Exception as ex:
        msg = str(ex)
        if 'is higher than' in msg and 'converter support' in msg:
            # Not a real failure — skl2onnx caps at a lower opset than
            # requested. The next iteration of this loop will succeed.
            import re as _re
            m = _re.search(r'support[^()]*\((\d+)\)', msg)
            cap = m.group(1) if m else 'lower'
            print(f'  ONNX opset {opset} not supported by this converter (max {cap}); using lower')
        else:
            print(f'  ONNX opset {opset} failed ({type(ex).__name__}: {ex}); trying lower')
if onnx_model is None:
    raise RuntimeError('All opset attempts failed.')

onnx.save_model(onnx_model, 'model.onnx')
print(f'Saved model.onnx ({len(onnx_model.SerializeToString()) / 1024:.1f} KB)')

## 8. Write sidecar file

In [ ]:
import json

# Trainer-aligned shape (matches ignode-trainer-tabular sidecars.py):
# {schema_version, feature_columns} + IGNODE-specific label_columns (used
# by the Custom Model Upload wizard to populate the Target Column field).
feature_sidecar = {'schema_version': 1, 'feature_columns': feature_columns, 'label_columns': [LABEL_COLUMN]}
with open('feature_columns.json', 'w') as f:
    json.dump(feature_sidecar, f, indent=2)

print('feature_columns.json:')
print(json.dumps(feature_sidecar, indent=2))

## 9. Download

In [ ]:
import zipfile
from datetime import datetime
from google.colab import files

# Distinct per-notebook + timestamped so multiple downloads don't
# overwrite each other and the file name identifies which notebook
# produced it. Format: ignode-tabular-regression-random-forest-YYYYMMDD-HHMM.zip
ZIP_NAME = f'ignode-tabular-regression-random-forest-{datetime.now().strftime("%Y%m%d-%H%M")}.zip'

# Bundle every artifact into one ZIP so the customer only clicks Download once.
ARTIFACTS = ['model.onnx', 'feature_columns.json']

with zipfile.ZipFile(ZIP_NAME, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in ARTIFACTS:
        zf.write(fname)

print(f'Bundled {len(ARTIFACTS)} files into {ZIP_NAME}')
files.download(ZIP_NAME)

## 10. Upload to IGNODE

1. **Integrations → ML Factory** in your IGNODE portal
2. Switch to the **Custom Models** tab
3. Click **+ Upload ML Model**
4. Drop your `model.onnx` and fill in the metadata form:
    - **Task Type:** `Regression`
    - **Feature Columns:** paste from `feature_columns.json` → `feature_columns`
    - **Target Column:** paste from `feature_columns.json` → `label_columns` (single entry)
5. Click **Upload**, then **Open in Playground** to test.

---

## Reusing this notebook for your own data

Two edits switch to your own dataset:

```python
# In the "Load data" cell:
SAMPLE_DATASET = None        # was 'equipment_rul_regression'

# In the "Settings" cell:
LABEL_COLUMN = 'YourColumn'  # was 'RemainingLife' — your NUMERIC target column
```

Everything else adapts automatically.

### Optional tweaks

| Want to change | Edit (Settings cell) |
|---|---|
| Forest size | `N_ESTIMATORS = 500` (more trees = more stable, slower) |
| Limit depth | `MAX_DEPTH = 10` (None = unlimited; lower = less overfitting) |
| Leaf minimum | `MIN_SAMPLES_LEAF = 5` (higher = less overfitting) |
| Train/test ratio | `TEST_SIZE = 0.3` |
| Reproducibility | `RANDOM_SEED = <any int>` |

### When to pick RandomForest over LightGBM / XGBoost

- **Small datasets** (< 1000 rows) — RandomForest is robust where boosting can overfit
- **You want interpretability** — the feature-importance ranking tells you what the model leans on
- **Reproducibility matters** — RandomForest is very stable across runs with the same `RANDOM_SEED`

### Common errors

| Error | Fix |
|---|---|
| `Label column must be numeric for regression` | Use the classifier notebook instead |
| `Label column 'X' not in CSV` | Check the column list printed by the inspect cell |
| Upload rejected: "invalid column names" | Re-export your CSV with the renamed columns shown in the prep cell |
| Overfitting (test R² much lower than train) | Raise `MIN_SAMPLES_LEAF` to 5 or 10, or lower `MAX_DEPTH` to 10-15 |